In [1]:
import os
import json
import torch
import gc
import time
import sys
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv, find_dotenv

_env_file = find_dotenv()
REPO_DIR = Path(_env_file).parent.resolve()
load_dotenv(_env_file)
ROOT = (REPO_DIR / os.getenv("ROOT")).resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.experiment_notebook import setup_experiment, fine_tune_and_evaluate
from src.utils.utils import set_all_seeds
from src.fairness_losses import get_fairness_loss

In [ ]:
experiment = "20260620_015632"  # "" = nou experiment | "20260414_153000" = reprendre un existent

In [ ]:
# ── Configuració del sweep ──────────────────────────────────────────────────
N_RUNS          = 20                          # models per configuració
BASE_SEED       = 42                         # seeds: BASE_SEED, BASE_SEED+1, …

ALPHAS        = [2, 4, 6, 8, 10]
LAYER_SIZES   = [4, 8, 12, 16, 20, 24, 28, 32, 48, 64, 80, 96, 112, 128]

MAX_EPOCHS    = 100   # màxim d'epochs per run
PATIENCE      = 10    # early stopping: atura si val_loss no millora en N epochs
BATCH_SIZE    = 32

print(f"Quantitat de runs: {len(ALPHAS) * len(LAYER_SIZES) * N_RUNS}")
FAIRNESS_LOSS = "tpr_parity"
DATASET       = "CheXpert-v1.0-small_raw"
IMG_SUBDIR    = ""

# Atribut sensible: 'gender' | 'age_group' | 'view_pos_binary'
SENSITIVE_ATTR = "age_group"
GROUP_LABELS   = {0: "Young", 1: "Senior"}

In [ ]:
experiment = "";  DATASET = "CheXpert-v1.0-small_raw";  IMG_SUBDIR = ""
SENSITIVE_ATTR = "age_group";  GROUP_LABELS = {0: "Young", 1: "Senior"}
ALPHAS = [2, 4, 6, 8, 10]
LAYER_SIZES = [4, 8, 12, 16, 20, 24, 28, 32, 48];  N_RUNS = 20

In [5]:
# ── Gestió de l'experiment ──────────────────────────────────────────────────
RESULTS_BASE = REPO_DIR / "results" / "sweep_bias_accuracy"
RESULTS_BASE.mkdir(parents=True, exist_ok=True)

if experiment == "":
    EXP_NAME = datetime.now().strftime("%Y%m%d_%H%M%S")
    EXP_DIR = RESULTS_BASE / EXP_NAME
    EXP_DIR.mkdir(parents=True, exist_ok=True)

    config = {
        "N_RUNS": N_RUNS,
        "BASE_SEED": BASE_SEED,
        "ALPHAS": ALPHAS,
        "LAYER_SIZES": LAYER_SIZES,
        "MAX_EPOCHS": MAX_EPOCHS,
        "PATIENCE": PATIENCE,
        "BATCH_SIZE": BATCH_SIZE,
        "FAIRNESS_LOSS": FAIRNESS_LOSS,
        "DATASET": DATASET,
        "IMG_SUBDIR": IMG_SUBDIR,
        "SENSITIVE_ATTR": SENSITIVE_ATTR,
        "GROUP_LABELS": {str(k): v for k, v in GROUP_LABELS.items()},
    }
    with open(EXP_DIR / "config.json", "w") as f:
        json.dump(config, f, indent=2)
    print(f"Nou experiment: {EXP_NAME}")

else:
    EXP_DIR = RESULTS_BASE / experiment
    if not EXP_DIR.exists():
        available = [d.name for d in RESULTS_BASE.iterdir() if d.is_dir()]
        raise FileNotFoundError(
            f"No s'ha trobat l'experiment '{experiment}' a {RESULTS_BASE}\n"
            f"Experiments disponibles: {available}"
        )
    with open(EXP_DIR / "config.json") as f:
        config = json.load(f)
    N_RUNS        = config["N_RUNS"]
    BASE_SEED     = config["BASE_SEED"]
    ALPHAS        = config["ALPHAS"]
    LAYER_SIZES   = config["LAYER_SIZES"]
    MAX_EPOCHS    = config["MAX_EPOCHS"]
    PATIENCE      = config["PATIENCE"]
    BATCH_SIZE    = config["BATCH_SIZE"]
    FAIRNESS_LOSS = config.get("FAIRNESS_LOSS", "tpr_parity")
    DATASET       = config.get("DATASET", "NIH_Chest_XRAY")
    IMG_SUBDIR    = config.get("IMG_SUBDIR", "sel_images")
    SENSITIVE_ATTR = config.get("SENSITIVE_ATTR", "gender")
    GROUP_LABELS   = {int(k): v for k, v in config.get("GROUP_LABELS", {"0": "Male", "1": "Female"}).items()}
    print(f"Reprenent experiment: {experiment}")
    print(f"Config carregada: {config}")

RESULTS_FILE = str(EXP_DIR / "sweep_results_v2.json")
print(f"\nDirectori : {EXP_DIR}")
print(f"Resultats : {RESULTS_FILE}")

Nou experiment: 20260612_015952

Directori : /home/jmassalle/Fairness-aware-CNN-pneumonia-prediction/results/sweep_bias_accuracy/20260612_015952
Resultats : /home/jmassalle/Fairness-aware-CNN-pneumonia-prediction/results/sweep_bias_accuracy/20260612_015952/sweep_results_v2.json


In [ ]:
# ── Sweep principal ─────────────────────────────────────────────────────────
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE, 'r') as f:
        all_results = json.load(f)
    print(f"Carregats {len(all_results)} runs previs de '{RESULTS_FILE}'")
else:
    all_results = []
    print("Fitxer de resultats nou.")

done = {(r['alpha'], r['ls'], r['run_id']) for r in all_results}

total_configs = len(ALPHAS) * len(LAYER_SIZES) * N_RUNS
remaining     = total_configs - len(done)
print(f"Total configuracions: {total_configs}  |  Pendents: {remaining}")

g0_name = GROUP_LABELS[0]
g1_name = GROUP_LABELS[1]

RETRY_WAIT_SECONDS = 15 * 60  # 15 minuts

# run_id és el bucle exterior: primer es completa 1 run per totes les combinacions,
# després una 2a seed per totes, etc. Així s'obtenen resultats parcials de tot el grid
# tan aviat com sigui possible.
for run_id in range(N_RUNS):
    for alpha in ALPHAS:
        for ls in LAYER_SIZES:
            if (alpha, ls, run_id) in done:
                print(f"[SKIP] alpha={alpha}, ls={ls}, run={run_id}")
                continue

            seed = BASE_SEED + run_id
            attempt = 0

            while True:
                attempt += 1
                print(f"\n{'='*60}")
                print(f"alpha={alpha}, ls={ls}, run={run_id}/{N_RUNS-1}, seed={seed}" +
                      (f"  [intent {attempt}]" if attempt > 1 else ""))
                print('='*60)

                try:
                    ctx = setup_experiment(
                        alpha=alpha,
                        prev2last_layer_size=ls,
                        batch_size=BATCH_SIZE,
                        seed=seed,
                        early_stopping_patience=PATIENCE,
                        run_name=f"sweep_a{alpha}_ls{ls}_r{run_id}",
                        results_base_dir=EXP_DIR / f"results_{DATASET}",
                        fairness_loss_fn=get_fairness_loss(FAIRNESS_LOSS),
                        data_subdir=f"data/{DATASET}",
                        img_subdir=IMG_SUBDIR,
                        sensitive_attr=SENSITIVE_ATTR,
                        group_labels=GROUP_LABELS,
                    )
                    set_all_seeds(seed)

                    result = fine_tune_and_evaluate(ctx, epochs=MAX_EPOCHS, alpha=alpha)

                    cm_g0 = result[g0_name]['confusion_matrix']
                    tpr_g0 = cm_g0[1][1] / (cm_g0[1][1] + cm_g0[1][0]) if (cm_g0[1][1] + cm_g0[1][0]) > 0 else float('nan')

                    cm_g1 = result[g1_name]['confusion_matrix']
                    tpr_g1 = cm_g1[1][1] / (cm_g1[1][1] + cm_g1[1][0]) if (cm_g1[1][1] + cm_g1[1][0]) > 0 else float('nan')

                    tpr_gap = abs(tpr_g0 - tpr_g1)

                    gender_pred = result.get('gender_pred', {})
                    last_layer_key = sorted(
                        [k for k in gender_pred.keys() if k.startswith('layer_')],
                        key=lambda x: int(x.split('_')[1])
                    )[-1] if gender_pred else None
                    sensitive_auc_last = gender_pred[last_layer_key]['auc'] if last_layer_key else float('nan')

                    run_result = {
                        'alpha'                   : alpha,
                        'ls'                      : ls,
                        'run_id'                  : run_id,
                        'seed'                    : seed,
                        'epochs_trained'          : ctx['total_epochs'],
                        'accuracy'                : result['All']['accuracy'],
                        'auc_all'                 : result['All']['auc'],
                        f'tpr_{g0_name.lower()}'  : tpr_g0,
                        f'tpr_{g1_name.lower()}'  : tpr_g1,
                        'tpr_gap'                 : tpr_gap,
                        'sensitive_auc_last_layer': sensitive_auc_last,
                        'sensitive_attr'          : SENSITIVE_ATTR,
                        'results_dir'             : ctx['results_dir'],
                    }
                    all_results.append(run_result)
                    done.add((alpha, ls, run_id))

                    with open(RESULTS_FILE, 'w') as f:
                        json.dump(all_results, f, indent=2)
                    print(f"  Run guardat ({ctx['total_epochs']} epochs) a '{RESULTS_FILE}'")
                    break

                except Exception as e:
                    if 'out of memory' in str(e).lower():
                        print(f"  [OOM] CUDA sense memòria (intent {attempt}): {e}")
                        gc.collect()
                        torch.cuda.empty_cache()
                        print(f"  Esperant {RETRY_WAIT_SECONDS // 60} minuts abans de reintentar...")
                        time.sleep(RETRY_WAIT_SECONDS)
                    else:
                        print(f"  ERROR inesperat: {e}")
                        import traceback; traceback.print_exc()
                        break

                finally:
                    if 'ctx' in dir():
                        del ctx
                    gc.collect()
                    torch.cuda.empty_cache()

print("\nSweep completat!")